In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import MemorySaver

In [2]:
def delete_name_tool(name: str):
    """ mock function to delete customer record from customer table based on the name"""
    query = f'delete from customer where name= {name}'
    return query

In [3]:
def send_email(email_id: str, content: str):
    """Mock function to send email"""
    print(content)
    return content

In [4]:
def read_email(email_address: str):
    """ mock email to read the email"""
    return "successfully read the email"

In [5]:
checkpointer = MemorySaver()

In [6]:
def handle_agent_response(result):
    # 1. Check for active interrupts
    if result.interrupts:
        print(f"\n{'='*10} ACTION REQUIRED {'='*10}")
        
        for interrupt in result.interrupts:
            # An interrupt can contain multiple action requests (parallel tool calls)
            for action in interrupt.value.get('action_requests', []):
                tool_name = action.get('name')
                tool_args = action.get('args', {})
                
                print(f"\n[Pending Tool]: {tool_name}")
                
                if tool_name == "delete_name_tool":
                    name = tool_args.get('name', 'UNKNOWN')
                    print(f" > SQL PREVIEW: DELETE FROM customer WHERE name = '{name}'")
                
                elif tool_name == "send_email":
                    content = tool_args.get('content', '')
                    recipient = tool_args.get('email_id', 'Unknown')
                    print(f" > TO: {recipient}")
                    print(f" > CONTENT: {content}")

            # Check allowed decisions from the first review config
            if interrupt.value.get('review_configs'):
                options = interrupt.value['review_configs'][0].get('allowed_decisions')
                print(f"\nYour Options: {options}")
        
        return "WAITING_FOR_USER_APPROVAL"

    # 2. Final Response Handling
    else:
        messages = result.value.get('messages', [])
        
        # Check if the last action was a tool execution (like your edit)
        # We look back to see if a ToolMessage exists in the last few messages
        for msg in reversed(messages):
            if msg.type == 'tool':
                print(f"--- TOOL EXECUTED (ACTUAL DATA) ---")
                print(f"Result: {msg.content}")
                break 

        if messages:
            last_message = messages[-1]
            print("\n--- FINAL AGENT RESPONSE ---")
            print(last_message.content)
            return "TASK_COMPLETED"
        
        return "NO_MESSAGES_FOUND"

In [7]:
agent = create_agent(
    model='gpt-4o-mini',
    tools=[delete_name_tool, send_email, read_email],
    checkpointer= checkpointer,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": {
                    "allowed_decisions": ["approve", "reject",'edit'],
                    "description": "Please review the  content  before sending email"
                },
                "delete_name_tool": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": """Please review the following SQL command before execution :
                    """
                },
                "read_email": False
                
            },
            description_prefix="Tool execution pending for approval"
        )
    ]
)

In [8]:
config = {"configurable": {"thread_id": "test"}}

In [9]:
result1 = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "delete all record from customer table with name as naji"
        }]
    },
    config=config,
    version="v2"
)

In [10]:
handle_agent_response(result=result1)


========== ACTION REQUIRED ==========

[Pending Tool]: delete_name_tool
 > SQL PREVIEW: DELETE FROM customer WHERE name = 'naji'

Your Options: ['approve', 'reject']


'WAITING_FOR_USER_APPROVAL'

In [11]:
from langgraph.types import Command


In [12]:
result2 = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config,
    version='v2'
)

In [13]:
handle_agent_response(result=result2)

--- TOOL EXECUTED (ACTUAL DATA) ---
Result: delete from customer where name= naji

--- FINAL AGENT RESPONSE ---
All records from the customer table with the name "naji" have been deleted successfully.


'TASK_COMPLETED'

In [14]:
result3 = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "send a email to naji@gmail.com saying that I am on leave on Tuesday "
        }]
    },
    config=config,
    version="v2"
)

In [15]:
handle_agent_response(result=result3)


========== ACTION REQUIRED ==========

[Pending Tool]: send_email
 > TO: naji@gmail.com
 > CONTENT: I am on leave on Tuesday.

Your Options: ['approve', 'reject', 'edit']


'WAITING_FOR_USER_APPROVAL'

In [16]:
result4 = agent.invoke(
    Command(
        resume={"decisions": [
            {"type": "edit",
             "edited_action": {
                 "name": "send_email",
                 "args": {
                     "email_id": "naji@gmail.com",
                     "content": "I will be on leave from November 22nd, 2026 to November 29th, 2026. I am currently on leave today, Tuesday."
                 }
             }}
        ]}
    ),
    config=config,
    version="v2"
)

I will be on leave from November 22nd, 2026 to November 29th, 2026. I am currently on leave today, Tuesday.


In [17]:
handle_agent_response(result=result4)

--- TOOL EXECUTED (ACTUAL DATA) ---
Result: I will be on leave from November 22nd, 2026 to November 29th, 2026. I am currently on leave today, Tuesday.

--- FINAL AGENT RESPONSE ---
The email has been sent to naji@gmail.com, informing them that you are on leave today, Tuesday.


'TASK_COMPLETED'

In [18]:
result5 = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "read email"
        }]
    },
    config=config,
    version="v2"
)

In [19]:
result5

GraphOutput(value={'messages': [HumanMessage(content='delete all record from customer table with name as naji', additional_kwargs={}, response_metadata={}, id='677a9c6b-1ca3-410d-b382-55e694dee6a1'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 110, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9a00d2658e', 'id': 'chatcmpl-DTW9Ga2ZOPVLfahEAe9z4eVAkTYmT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7d86-87f3-7002-8fd5-1179fdf602bc-0', tool_calls=[{'name': 'delete_name_tool', 'args': {'name': 'naji'}, 'id': 'call_nmE1mwrHMGUkhSeiXMDc7GAP', 'type': 'tool_call'}], invalid_tool_calls

In [20]:
handle_agent_response(result5)

--- TOOL EXECUTED (ACTUAL DATA) ---
Result: successfully read the email

--- FINAL AGENT RESPONSE ---
The email has been successfully read. If you need any further assistance or specific details, feel free to ask!


'TASK_COMPLETED'